In [1]:
!pip install transformers datasets seqeval accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!wget https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/train.txt
!wget https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/valid.txt
!wget https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/test.txt

--2026-04-06 04:36:07--  https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-06 04:36:07 ERROR 404: Not Found.

--2026-04-06 04:36:07--  https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/valid.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-06 04:36:08 ERROR 404: Not Found.

--2026-04-06 04:36:08--  https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/test.txt
Resolving raw.gith

In [3]:
def read_conll(file_path):
    sentences = []
    labels = []

    with open(file_path, "r") as f:
        tokens = []
        tags = []

        for line in f:
            if line.strip() == "":
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.strip().split()
                tokens.append(parts[0])
                tags.append(parts[-1])  # last column = tag

    return sentences, labels

import os

# Check if files exist or are empty, if so, download them
file_downloaded_train = False
if not os.path.exists("train.txt") or os.path.getsize("train.txt") == 0:
    print("train.txt not found or empty, attempting download...")
    !wget -O train.txt https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train
    if os.path.exists("train.txt") and os.path.getsize("train.txt") > 0:
        file_downloaded_train = True
else:
    print("train.txt already exists and is not empty.")

file_downloaded_valid = False
if not os.path.exists("valid.txt") or os.path.getsize("valid.txt") == 0:
    print("valid.txt not found or empty, attempting download...")
    !wget -O valid.txt https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa
    if os.path.exists("valid.txt") and os.path.getsize("valid.txt") > 0:
        file_downloaded_valid = True
else:
    print("valid.txt already exists and is not empty.")

# Diagnostic checks
print(f"train.txt exists: {os.path.exists('train.txt')}, size: {os.path.getsize('train.txt') if os.path.exists('train.txt') else 'N/A'}")
print(f"valid.txt exists: {os.path.exists('valid.txt')}, size: {os.path.getsize('valid.txt') if os.path.exists('valid.txt') else 'N/A'}")

train_tokens, train_labels = read_conll("train.txt")
val_tokens, val_labels = read_conll("valid.txt")

print(f"Number of training sentences: {len(train_tokens)}")
print(f"Number of training labels: {len(train_labels)}")
print(f"Number of validation sentences: {len(val_tokens)}")
print(f"Number of validation labels: {len(val_labels)}")

train.txt not found or empty, attempting download...
--2026-04-06 04:36:11--  https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3281528 (3.1M) [text/plain]
Saving to: ‘train.txt’

train.txt           100%[===================>]   3.13M  --.-KB/s    in 0.05s   

2026-04-06 04:36:11 (68.5 MB/s) - ‘train.txt’ saved [3281528/3281528]

valid.txt not found or empty, attempting download...
--2026-04-06 04:36:11--  https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.11

In [4]:
unique_labels = list(set(tag for sent in train_labels for tag in sent))

label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(tokens, labels):
    tokenized = tokenizer(tokens, is_split_into_words=True, truncation=True)

    word_ids = tokenized.word_ids()
    aligned_labels = []
    prev = None

    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != prev:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            aligned_labels.append(-100)
        prev = word_idx

    tokenized["labels"] = aligned_labels
    return tokenized


train_encodings = [tokenize_and_align(t, l) for t, l in zip(train_tokens[:2000], train_labels[:2000])]
val_encodings = [tokenize_and_align(t, l) for t, l in zip(val_tokens[:500], val_labels[:500])]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
import torch

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {k: torch.tensor(v) for k, v in self.encodings[idx].items()}

    def __len__(self):
        return len(self.encodings)


train_dataset = CustomDataset(train_encodings)
val_dataset = CustomDataset(val_encodings)

In [7]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs=2,   # fast
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.130592,0.089476
2,0.051045,0.068463


TrainOutput(global_step=250, training_loss=0.15649484348297119, metrics={'train_runtime': 1032.1777, 'train_samples_per_second': 3.875, 'train_steps_per_second': 0.242, 'total_flos': 46987135146240.0, 'train_loss': 0.15649484348297119, 'epoch': 2.0})

In [13]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=2)

    word_ids = inputs.word_ids()
    result = []
    prev = None

    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == prev:
            continue
        result.append((tokens[word_idx], id2label[preds[0][i].item()]))
        prev = word_idx

    return result

print(predict("John works at Google in California"))

[('John', 'I-PER'), ('works', 'O'), ('at', 'O'), ('Google', 'I-ORG'), ('in', 'O'), ('California', 'I-LOC')]
